# Average a metric across superclass CSVs

Reads every `*.csv` in `analysis/data/resnet50`, extracts one column, and averages it per `(Tau, m)` combination.

In [ ]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path("data/resnet50")
METRIC = "val_test_genus_knn_top1"  # <- change to the column you want

files = sorted(DATA_DIR.glob("*.csv"))
print(f"{len(files)} files:", [f.stem for f in files])

In [ ]:
# Available metric columns (from the first file)
list(pd.read_csv(files[0]).columns)

In [ ]:
def load(path: Path, metric: str) -> pd.DataFrame:
    df = pd.read_csv(path, dtype={"Tau": str, "m": str})
    if metric not in df.columns:
        raise KeyError(f"{metric!r} not in {path.name}")
    df = df[["Tau", "m", metric]].copy()
    df["Tau"] = df["Tau"].fillna("-").replace("", "-")
    df["m"] = df["m"].fillna("-").replace("", "-")
    df[metric] = pd.to_numeric(df[metric], errors="coerce")
    df["source"] = path.stem.split("-")[0]
    return df


long = pd.concat([load(f, METRIC) for f in files], ignore_index=True)
long.head()

In [ ]:
# Per-file values plus the mean, indexed by (Tau, m)
table = long.pivot_table(index=["Tau", "m"], columns="source", values=METRIC, aggfunc="mean")
table["mean"] = table.mean(axis=1)
table["n_files"] = long.groupby(["Tau", "m"])[METRIC].count()


def sort_key(value: str) -> float:
    try:
        return float(value)
    except ValueError:
        return float("-inf")


table = table.sort_index(key=lambda idx: idx.map(sort_key), ascending=False)
table.round(4)

In [ ]:
# Compact table: Tau and m as separate columns
summary = table[["mean", "n_files"]].reset_index()[["Tau", "m", "mean", "n_files"]]
summary.round(4)


In [ ]:
out = DATA_DIR.parent / f"avg_{METRIC}_by_tau_m.csv"
table.round(6).to_csv(out)
print("wrote", out)